# Plegamiento de Proteínas con Algoritmos Genéticos

Cuaderno listo para **Google Colab** o **JupyterLab**.

Incluye:
- modelo HP simplificado en 2D,
- cromosoma basado en movimientos relativos,
- decodificación a coordenadas,
- detección de solapamientos,
- función fitness basada en contactos H-H,
- gráficos del plegamiento final.


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt


## Datos del problema


In [ ]:
secuencia = 'HPPHHPHPH'
n = len(secuencia)

tam_poblacion = 100
n_generaciones = 150
prob_cruce = 0.8
prob_mutacion = 0.08
tam_torneo = 3

random.seed(42)
np.random.seed(42)


## Direcciones y representación


In [ ]:
direcciones = {
    0: (1, 0),
    1: (0, 1),
    2: (-1, 0),
    3: (0, -1)
}

def girar(direccion_actual, movimiento):
    if movimiento == 0:
        return direccion_actual
    elif movimiento == 1:
        return (direccion_actual + 1) % 4
    else:
        return (direccion_actual - 1) % 4

def crear_individuo():
    return np.random.randint(0, 3, size=n - 2)


## Decodificación y fitness


In [ ]:
def decodificar(individuo):
    coords = [(0, 0), (1, 0)]
    direccion_actual = 0
    ocupadas = {(0, 0), (1, 0)}
    solapamiento = False
    x, y = 1, 0

    for mov in individuo:
        direccion_actual = girar(direccion_actual, mov)
        dx, dy = direcciones[direccion_actual]
        x, y = x + dx, y + dy
        if (x, y) in ocupadas:
            solapamiento = True
        coords.append((x, y))
        ocupadas.add((x, y))

    return coords, solapamiento

def contactos_hh(coords, secuencia):
    contactos = 0
    for i in range(len(coords)):
        for j in range(i + 2, len(coords)):
            if secuencia[i] == 'H' and secuencia[j] == 'H':
                x1, y1 = coords[i]
                x2, y2 = coords[j]
                dist = abs(x1 - x2) + abs(y1 - y2)
                if dist == 1:
                    contactos += 1
    return contactos

def fitness(individuo):
    coords, solapamiento = decodificar(individuo)
    contactos = contactos_hh(coords, secuencia)
    if solapamiento:
        return -100 + contactos
    return contactos


## Operadores genéticos


In [ ]:
def seleccion_torneo(poblacion):
    candidatos = random.sample(poblacion, tam_torneo)
    return max(candidatos, key=fitness).copy()

def cruce_un_punto(padre1, padre2):
    if random.random() < prob_cruce:
        punto = random.randint(1, len(padre1) - 1)
        hijo1 = np.concatenate([padre1[:punto], padre2[punto:]])
        hijo2 = np.concatenate([padre2[:punto], padre1[punto:]])
        return hijo1, hijo2
    return padre1.copy(), padre2.copy()

def mutar(individuo):
    mutado = individuo.copy()
    for i in range(len(mutado)):
        if random.random() < prob_mutacion:
            opciones = [0, 1, 2]
            opciones.remove(mutado[i])
            mutado[i] = random.choice(opciones)
    return mutado


## Ejecución del algoritmo genético


In [ ]:
poblacion = [crear_individuo() for _ in range(tam_poblacion)]

mejores_fitness = []
promedios_fitness = []
mejor_individuo_global = None
mejor_fit_global = -float('inf')

for generacion in range(n_generaciones):
    nueva_poblacion = []
    while len(nueva_poblacion) < tam_poblacion:
        padre1 = seleccion_torneo(poblacion)
        padre2 = seleccion_torneo(poblacion)
        hijo1, hijo2 = cruce_un_punto(padre1, padre2)
        hijo1 = mutar(hijo1)
        hijo2 = mutar(hijo2)
        nueva_poblacion.append(hijo1)
        if len(nueva_poblacion) < tam_poblacion:
            nueva_poblacion.append(hijo2)
    poblacion = nueva_poblacion
    fitness_poblacion = [fitness(ind) for ind in poblacion]
    mejor_generacion = max(poblacion, key=fitness)
    mejor_fit = fitness(mejor_generacion)
    promedio_fit = np.mean(fitness_poblacion)
    mejores_fitness.append(mejor_fit)
    promedios_fitness.append(promedio_fit)
    if mejor_fit > mejor_fit_global:
        mejor_fit_global = mejor_fit
        mejor_individuo_global = mejor_generacion.copy()

coords_final, solapamiento_final = decodificar(mejor_individuo_global)
contactos_final = contactos_hh(coords_final, secuencia)

print('Mejor cromosoma encontrado:', mejor_individuo_global.tolist())
print('Fitness final:', mejor_fit_global)
print('Contactos H-H:', contactos_final)
print('¿Hay solapamiento?:', solapamiento_final)
print('Coordenadas:', coords_final)


## Curva de convergencia


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(mejores_fitness, label='Mejor fitness')
plt.plot(promedios_fitness, label='Fitness promedio')
plt.xlabel('Generación')
plt.ylabel('Fitness')
plt.title('Evolución del algoritmo genético')
plt.legend()
plt.grid(True)
plt.show()


## Plegamiento final


In [ ]:
plt.figure(figsize=(7, 7))
for i in range(len(coords_final) - 1):
    x1, y1 = coords_final[i]
    x2, y2 = coords_final[i + 1]
    plt.plot([x1, x2], [y1, y2], linewidth=2)

for i, (x, y) in enumerate(coords_final):
    color = 'red' if secuencia[i] == 'H' else 'lightblue'
    plt.scatter(x, y, s=500, c=color, edgecolors='black')
    plt.text(x, y, f'{secuencia[i]}{i}', ha='center', va='center', fontsize=9)

plt.title('Mejor plegamiento encontrado')
plt.axis('equal')
plt.grid(True)
plt.show()


## Contactos H-H


In [ ]:
plt.figure(figsize=(7, 7))
for i in range(len(coords_final) - 1):
    x1, y1 = coords_final[i]
    x2, y2 = coords_final[i + 1]
    plt.plot([x1, x2], [y1, y2], linewidth=2)

for i, (x, y) in enumerate(coords_final):
    color = 'red' if secuencia[i] == 'H' else 'lightblue'
    plt.scatter(x, y, s=500, c=color, edgecolors='black')
    plt.text(x, y, f'{secuencia[i]}{i}', ha='center', va='center', fontsize=9)

for i in range(len(coords_final)):
    for j in range(i + 2, len(coords_final)):
        if secuencia[i] == 'H' and secuencia[j] == 'H':
            x1, y1 = coords_final[i]
            x2, y2 = coords_final[j]
            if abs(x1 - x2) + abs(y1 - y2) == 1:
                plt.plot([x1, x2], [y1, y2], linestyle='--', linewidth=1)

plt.title('Contactos H-H no consecutivos')
plt.axis('equal')
plt.grid(True)
plt.show()


## Ejercicio sugerido

Modificar:
- la secuencia HP,
- la cantidad de generaciones,
- la probabilidad de mutación,
- el tamaño de la población,

y analizar cómo cambia la calidad del plegamiento obtenido.
